# W4 — Streamlit Forecasting Notebook

This notebook is aligned with the previous `w3_streamlit_ona.ipynb` workflow and keeps the same saved artifact patterns:

- `outputs/timeseries_with_features.csv`
- `models/best_model.pkl`
- `models/feature_columns.json`
- `models/model_metrics.csv`

The goal is to move from a single notebook prediction to a cleaner Week 4 workflow that is ready for a Streamlit app.


---
## 1. Project setup

This section detects the project root, defines the expected file paths, and verifies that the notebook is using the same structure as the previous version.


In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Detect the project root.
# If the notebook is inside a notebooks/ folder, move one level up.
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "models").exists() and (CURRENT_DIR / "outputs").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists() and (CURRENT_DIR.parent / "outputs").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

MODELS_DIR = PROJECT_DIR / "models"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
APP_DIR = PROJECT_DIR / "app"

MODEL_PATH = MODELS_DIR / "best_model.pkl"
FEATURES_PATH = MODELS_DIR / "feature_columns.json"
METRICS_PATH = MODELS_DIR / "model_metrics.csv"
DATA_PATH = OUTPUTS_DIR / "timeseries_with_features.csv"

print("Project directory:", PROJECT_DIR)


In [ ]:
expected_files = {
    "Saved model": MODEL_PATH,
    "Feature list": FEATURES_PATH,
    "Model metrics": METRICS_PATH,
    "Feature dataset": DATA_PATH,
}

missing_files = []
for label, path in expected_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:16s} | {status:7s} | {path.relative_to(PROJECT_DIR) if path.exists() else path}")
    if not path.exists():
        missing_files.append(path)

if missing_files:
    raise FileNotFoundError(
        "Some required project files are missing. "
        "Check that this notebook is running from the project root or from a notebooks/ folder."
    )


---
## 2. Load saved artifacts

This section follows the same pattern as your previous notebook: load the trained model, feature column list, and saved model metrics.


In [ ]:
model = joblib.load(MODEL_PATH)

with open(FEATURES_PATH, "r") as f:
    FEATURE_COLUMNS = json.load(f)

metrics_df = pd.read_csv(METRICS_PATH)

print("Model loaded successfully")
print("Number of expected features:", len(FEATURE_COLUMNS))
display(metrics_df.head())


---
## 3. Load the feature dataset

This uses `outputs/timeseries_with_features.csv`, exactly like the previous notebook.


In [ ]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Data loaded:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
display(df.head())


In [ ]:
required_base_columns = {"date", "unit_sales"}
missing_base_columns = required_base_columns - set(df.columns)
if missing_base_columns:
    raise ValueError(f"Missing required columns: {missing_base_columns}")

missing_model_features = [col for col in FEATURE_COLUMNS if col not in df.columns]
print("Features expected by the model:", len(FEATURE_COLUMNS))
print("Features not directly present in the dataset:", len(missing_model_features))

if missing_model_features:
    print("These features will be created dynamically or filled with 0 when needed:")
    print(missing_model_features[:20])


---
## 4. Quick historical sales check

Before forecasting, inspect the historical target series. This helps confirm that the data looks consistent before it is used by the app.


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["unit_sales"], label="Historical unit sales")
plt.title("Historical Unit Sales")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
## 5. Feature builder for a target date

This keeps the core pattern from your previous notebook: `build_features_for_date(df, target_date)`.

Improvements added here:

- Validates that enough historical rows exist for lag and rolling features.
- Preserves the exact model feature order from `feature_columns.json`.
- Handles optional columns safely.
- Returns a clean one-row DataFrame ready for `model.predict()`.


In [ ]:
def build_features_for_date(history_df, target_date, feature_columns=FEATURE_COLUMNS):
    """Build one prediction row for a target date using the latest available history.

    Parameters
    ----------
    history_df : pandas.DataFrame
        Historical data with at least date and unit_sales columns.
    target_date : str or pandas.Timestamp
        Date to forecast.
    feature_columns : list
        Feature names expected by the trained model.

    Returns
    -------
    pandas.DataFrame
        One-row feature matrix ordered exactly as the trained model expects.
    """
    history_df = history_df.copy().sort_values("date").reset_index(drop=True)
    target_date = pd.Timestamp(target_date)

    max_lag = 30
    if len(history_df) < max_lag:
        raise ValueError(f"At least {max_lag} historical rows are required to build lag features.")

    features = {}

    # Calendar features.
    features["day"] = target_date.day
    features["month"] = target_date.month
    features["dayofweek"] = target_date.dayofweek
    features["is_weekend"] = int(target_date.dayofweek >= 5)
    features["week_of_year"] = int(target_date.isocalendar().week)

    # Lag features based on the latest available historical sales.
    for lag in [1, 7, 14, 30]:
        features[f"lag_{lag}"] = history_df["unit_sales"].iloc[-lag]

    # Rolling features based only on past values.
    features["rolling_7d_mean"] = history_df["unit_sales"].iloc[-7:].mean()
    features["rolling_14d_mean"] = history_df["unit_sales"].iloc[-14:].mean()
    features["rolling_30d_mean"] = history_df["unit_sales"].iloc[-30:].mean()
    features["rolling_7d_std"] = history_df["unit_sales"].iloc[-7:].std()

    # Optional external features from the latest known row.
    optional_latest_features = [
        "oil_lag_1",
        "oil_lag_7",
        "oil_rolling_7d_mean",
    ]
    for col in optional_latest_features:
        if col in history_df.columns:
            features[col] = history_df[col].iloc[-1]

    # Fill any remaining expected model features with 0.
    # This keeps the prediction function robust if the model was trained with encoded columns.
    for col in feature_columns:
        if col not in features:
            features[col] = 0

    X = pd.DataFrame([features])[feature_columns]
    return X


---
## 6. Single-date forecast

This reproduces the previous notebook behavior: generate one prediction for a selected target date.


In [ ]:
target_date = pd.Timestamp("2014-04-01")

X_new = build_features_for_date(df, target_date)
prediction = float(model.predict(X_new)[0])

print(f"Prediction for {target_date.date()}: {prediction:.2f} units")
display(X_new.head())


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["unit_sales"], label="Historical unit sales")
plt.scatter(target_date, prediction, label="Forecast", s=80)
plt.title("Single-Date Forecast")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
## 7. Multi-day forecast

This is the main Week 4 improvement. The function below forecasts several days ahead by feeding each prediction back into the history. This makes the notebook closer to a real Streamlit forecasting service.


In [ ]:
def forecast_next_days(history_df, start_date, horizon, model=model, feature_columns=FEATURE_COLUMNS):
    """Forecast multiple future days recursively.

    Each predicted value is appended to the temporary history and then used to build
    features for the next forecast date.
    """
    if horizon < 1:
        raise ValueError("horizon must be at least 1")

    temp_history = history_df.copy().sort_values("date").reset_index(drop=True)
    start_date = pd.Timestamp(start_date)
    forecasts = []

    for step in range(horizon):
        forecast_date = start_date + pd.Timedelta(days=step)
        X_step = build_features_for_date(temp_history, forecast_date, feature_columns)
        y_hat = float(model.predict(X_step)[0])

        forecasts.append({
            "date": forecast_date,
            "forecast": y_hat,
        })

        # Append the forecast as the next known value for recursive forecasting.
        new_row = {col: np.nan for col in temp_history.columns}
        new_row["date"] = forecast_date
        new_row["unit_sales"] = y_hat
        temp_history = pd.concat([temp_history, pd.DataFrame([new_row])], ignore_index=True)

    return pd.DataFrame(forecasts)


In [ ]:
forecast_horizon = 14
start_date = df["date"].max() + pd.Timedelta(days=1)

forecast_df = forecast_next_days(df, start_date=start_date, horizon=forecast_horizon)

display(forecast_df)


In [ ]:
history_window = 180
plot_history = df.tail(history_window)

plt.figure(figsize=(14, 5))
plt.plot(plot_history["date"], plot_history["unit_sales"], label="Historical unit sales")
plt.plot(forecast_df["date"], forecast_df["forecast"], marker="o", label="Forecast")
plt.title(f"{forecast_horizon}-Day Forecast")
plt.xlabel("Date")
plt.ylabel("Unit sales")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
## 8. Save forecast output

This creates a forecast CSV that can be used for reporting, testing, or as an app output.


In [ ]:
FORECAST_OUTPUT_PATH = OUTPUTS_DIR / "streamlit_forecast_output.csv"
forecast_df.to_csv(FORECAST_OUTPUT_PATH, index=False)
print("Forecast saved to:", FORECAST_OUTPUT_PATH.relative_to(PROJECT_DIR))


---
## 9. Generate a Streamlit app file

This cell creates `app/main.py`. The app uses the same artifacts as the notebook and allows the user to select a start date and forecast horizon.


In [ ]:
APP_DIR.mkdir(exist_ok=True)
APP_PATH = APP_DIR / "main.py"

streamlit_app_code = 'import json\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nst.set_page_config(page_title="Sales Forecasting App", layout="wide")\n\nPROJECT_DIR = Path(__file__).resolve().parents[1]\nMODELS_DIR = PROJECT_DIR / "models"\nOUTPUTS_DIR = PROJECT_DIR / "outputs"\n\nMODEL_PATH = MODELS_DIR / "best_model.pkl"\nFEATURES_PATH = MODELS_DIR / "feature_columns.json"\nMETRICS_PATH = MODELS_DIR / "model_metrics.csv"\nDATA_PATH = OUTPUTS_DIR / "timeseries_with_features.csv"\n\n\n@st.cache_resource\ndef load_model():\n    return joblib.load(MODEL_PATH)\n\n\n@st.cache_data\ndef load_features():\n    with open(FEATURES_PATH, "r") as f:\n        return json.load(f)\n\n\n@st.cache_data\ndef load_metrics():\n    if METRICS_PATH.exists():\n        return pd.read_csv(METRICS_PATH)\n    return pd.DataFrame()\n\n\n@st.cache_data\ndef load_data():\n    df = pd.read_csv(DATA_PATH)\n    df["date"] = pd.to_datetime(df["date"])\n    return df.sort_values("date").reset_index(drop=True)\n\n\ndef build_features_for_date(history_df, target_date, feature_columns):\n    history_df = history_df.copy().sort_values("date").reset_index(drop=True)\n    target_date = pd.Timestamp(target_date)\n\n    max_lag = 30\n    if len(history_df) < max_lag:\n        raise ValueError(f"At least {max_lag} historical rows are required to build lag features.")\n\n    features = {}\n    features["day"] = target_date.day\n    features["month"] = target_date.month\n    features["dayofweek"] = target_date.dayofweek\n    features["is_weekend"] = int(target_date.dayofweek >= 5)\n    features["week_of_year"] = int(target_date.isocalendar().week)\n\n    for lag in [1, 7, 14, 30]:\n        features[f"lag_{lag}"] = history_df["unit_sales"].iloc[-lag]\n\n    features["rolling_7d_mean"] = history_df["unit_sales"].iloc[-7:].mean()\n    features["rolling_14d_mean"] = history_df["unit_sales"].iloc[-14:].mean()\n    features["rolling_30d_mean"] = history_df["unit_sales"].iloc[-30:].mean()\n    features["rolling_7d_std"] = history_df["unit_sales"].iloc[-7:].std()\n\n    for col in ["oil_lag_1", "oil_lag_7", "oil_rolling_7d_mean"]:\n        if col in history_df.columns:\n            features[col] = history_df[col].iloc[-1]\n\n    for col in feature_columns:\n        if col not in features:\n            features[col] = 0\n\n    return pd.DataFrame([features])[feature_columns]\n\n\ndef forecast_next_days(history_df, start_date, horizon, model, feature_columns):\n    temp_history = history_df.copy().sort_values("date").reset_index(drop=True)\n    start_date = pd.Timestamp(start_date)\n    forecasts = []\n\n    for step in range(horizon):\n        forecast_date = start_date + pd.Timedelta(days=step)\n        X_step = build_features_for_date(temp_history, forecast_date, feature_columns)\n        y_hat = float(model.predict(X_step)[0])\n\n        forecasts.append({"date": forecast_date, "forecast": y_hat})\n\n        new_row = {col: np.nan for col in temp_history.columns}\n        new_row["date"] = forecast_date\n        new_row["unit_sales"] = y_hat\n        temp_history = pd.concat([temp_history, pd.DataFrame([new_row])], ignore_index=True)\n\n    return pd.DataFrame(forecasts)\n\n\nst.title("Sales Forecasting App")\nst.write("Forecast future unit sales using the trained model artifacts from the project.")\n\nmodel = load_model()\nfeature_columns = load_features()\nmetrics_df = load_metrics()\ndf = load_data()\n\nwith st.sidebar:\n    st.header("Forecast settings")\n    default_start = df["date"].max().date() + pd.Timedelta(days=1)\n    start_date = st.date_input("Forecast start date", value=default_start)\n    horizon = st.slider("Forecast horizon", min_value=1, max_value=30, value=14)\n    history_window = st.slider("Historical window shown", min_value=30, max_value=365, value=180)\n\ncol1, col2, col3 = st.columns(3)\ncol1.metric("Rows", f"{len(df):,}")\ncol2.metric("Last historical date", str(df["date"].max().date()))\ncol3.metric("Model features", len(feature_columns))\n\nif not metrics_df.empty:\n    st.subheader("Saved Model Metrics")\n    st.dataframe(metrics_df)\n\nforecast_df = forecast_next_days(df, start_date=start_date, horizon=horizon, model=model, feature_columns=feature_columns)\n\nst.subheader("Forecast Table")\nst.dataframe(forecast_df)\n\nplot_history = df.tail(history_window)\nfig, ax = plt.subplots(figsize=(12, 5))\nax.plot(plot_history["date"], plot_history["unit_sales"], label="Historical unit sales")\nax.plot(forecast_df["date"], forecast_df["forecast"], marker="o", label="Forecast")\nax.set_title(f"{horizon}-Day Unit Sales Forecast")\nax.set_xlabel("Date")\nax.set_ylabel("Unit sales")\nax.legend()\nax.grid(alpha=0.3)\nst.pyplot(fig)\n\ncsv = forecast_df.to_csv(index=False).encode("utf-8")\nst.download_button(\n    label="Download forecast as CSV",\n    data=csv,\n    file_name="forecast_output.csv",\n    mime="text/csv",\n)\n'

APP_PATH.write_text(streamlit_app_code, encoding="utf-8")
print("Streamlit app created at:", APP_PATH.relative_to(PROJECT_DIR))


---
## 10. How to run the app

From the project root, run:

```bash
python -m streamlit run app/main.py
```

If your notebook is inside a `notebooks/` folder, open a terminal from the project root before running the command.


---
## 11. Summary of improvements

This notebook keeps the same core pattern as `w3_streamlit_ona.ipynb`, but improves it by adding:

- Project root detection.
- File checks before loading artifacts.
- More robust feature validation.
- A reusable single-date forecast function.
- A recursive multi-day forecasting function.
- Forecast CSV export.
- Automatic generation of a production-style `app/main.py` Streamlit file.
- Consistent English text and comments throughout the notebook.
